# Observation-series + intelligence-report GraphSAGE → relation-aware HGT classification

This notebook is a copy of `observation_series_and_intel_rgcn_classification.ipynb` that replaces the residual r-GCN classifier with an advanced network: configurable-hop GraphSAGE-style neighbourhood message passing followed by relation-aware HGT multi-head attention layers.

The supervised targets remain observation-level aircraft variant, radar mode, radar type, and operator country. Ground-truth labels and synthetic report truth markers are stripped from model features; the report text-derived claim fields are treated as noisy external evidence.


In [1]:
from __future__ import annotations

import json, math, random, sys, time
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.checkpoint
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from esm_observation_series_generator import load_observation_series_json
from rgcn_fusion.intelligence_reports import report_claim_score, report_recency_score
DATA_PATH = ROOT / "generated" / "demo_esm_observation_series_with_intel.json"
ARTIFACT_DIR = ROOT / "artifacts" / "observation_series_and_intel_rgcn_classification_advanced_network"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_PATH)
print(ARTIFACT_DIR)
print(DEVICE)


C:\Users\theon\r-GCN_fusion\generated\demo_esm_observation_series_with_intel.json
C:\Users\theon\r-GCN_fusion\artifacts\observation_series_and_intel_rgcn_classification_advanced_network
cuda


## Load series and create a label-free inference view

Ground-truth fields are retained in a separate target table, but are recursively stripped from the feature payload before graph construction. This prevents leakage from `ground_truth_label`, `ground_truth_track_label`, and `ground_truth_mode_sequence` into node features or edges.


In [2]:
LEAKAGE_KEYS = {"ground_truth_label", "ground_truth_track_label", "ground_truth_mode_sequence", "synthetic_truth_value"}
TARGETS = ["aircraft_variant", "radar_mode", "radar_type", "operator_country"]

def strip_ground_truth(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: strip_ground_truth(v) for k, v in obj.items() if k not in LEAKAGE_KEYS}
    if isinstance(obj, list):
        return [strip_ground_truth(v) for v in obj]
    return obj

raw = load_observation_series_json(DATA_PATH)
series_records = raw["observation_series"]
inference_records = strip_ground_truth(series_records)

# Targets are extracted only from the original payload and never merged back into features.
target_rows = []
for s in series_records:
    for obs in s["observations"]:
        gt = obs["ground_truth_label"]
        target_rows.append({
            "observation_id": obs["observation_id"],
            "series_id": obs["series_id"],
            "sequence_index": obs["sequence_index"],
            "aircraft_variant": gt.get("aircraft_variant"),
            "radar_mode": gt.get("mode") or gt.get("radar_mode"),
            "radar_type": gt.get("radar") or gt.get("radar_type"),
            "operator_country": gt.get("operator") or gt.get("operator_country"),
        })

serialized_features = json.dumps(inference_records)
assert "ground_truth" not in serialized_features, "Ground-truth fields leaked into inference view"
report_count = sum(len(obs.get("intelligence_reports") or []) for series in series_records for obs in series["observations"])
print(f"series={len(series_records):,}, observations={len(target_rows):,}, intelligence_reports={report_count:,}")
print(target_rows[0])


series=2,500, observations=80,118, intelligence_reports=240,166
{'observation_id': 'esm_series_00001_obs_001', 'series_id': 'esm_series_00001', 'sequence_index': 0, 'aircraft_variant': 'F-16V Block 70/72', 'radar_mode': 'look_down_shoot_down', 'radar_type': 'AN/APG-83 SABR', 'operator_country': 'Slovakia'}


## Feature engineering from observations only

Features use measured ESM parameters, approximate kinematics, elapsed time, and sensor metadata from the label-free inference view. Candidate evidence is disabled by default so candidate-hypothesis fields such as candidate radar/mode cannot become proxy labels for the observation-level radar-mode target. Optional segment IDs are derived only from measured radar-parameter changes and are kept as numeric features; segment edges remain disabled by default for the cleanest benchmark.


In [3]:
INCLUDE_CANDIDATE_NODES = True
INCLUDE_INTEL_REPORT_NODES = True
USE_SEGMENT_EDGES = False
SEGMENT_FREQUENCY_SHIFT_MHZ = 750.0


def flatten_numeric(prefix: str, value: Any, out: dict[str, float]) -> None:
    if isinstance(value, dict):
        for k, v in value.items():
            flatten_numeric(f"{prefix}.{k}" if prefix else k, v, out)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        out[prefix] = float(value)


def _numeric_value(value: Any) -> float | None:
    if isinstance(value, dict):
        value = value.get("value")
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    return None


def segment_indices(observations: list[dict[str, Any]]) -> list[int]:
    """Infer coarse transition markers from measured frequency only, never labels/candidates."""
    segs, current = [], 0
    prev_freq = None
    for obs in observations:
        freq = _numeric_value(obs.get("esm_radar_parameters", {}).get("frequency_mhz"))
        shift = prev_freq is not None and freq is not None and abs(freq - prev_freq) > SEGMENT_FREQUENCY_SHIFT_MHZ
        if segs and shift:
            current += 1
        segs.append(current)
        prev_freq = freq
    return segs


feature_rows, node_meta = [], []
observation_node_indices: list[int] = []
candidates_by_observation_node: dict[int, list[int]] = defaultdict(list)
reports_by_observation_node: dict[int, list[int]] = defaultdict(list)
claims_by_observation_node: dict[int, list[int]] = defaultdict(list)
claims_by_claim_type: dict[tuple[str, str], list[int]] = defaultdict(list)
observation_time_by_node: dict[int, datetime | None] = {}
pending_report_nodes: list[tuple[int, dict[str, Any], int]] = []
pending_candidate_nodes: list[tuple[int, dict[str, Any], int, int]] = []
for series in inference_records:
    obs_list = sorted(series["observations"], key=lambda o: o["sequence_index"])
    segs = segment_indices(obs_list)
    n = max(len(obs_list), 1)
    for obs, seg in zip(obs_list, segs):
        obs_node_idx = len(feature_rows)
        observation_node_indices.append(obs_node_idx)
        feats = {
            "node_kind_observation": 1.0,
            "node_kind_candidate": 0.0,
            "elapsed_time_s": float(obs.get("elapsed_time_s", 0.0)),
            "sequence_fraction": float(obs.get("sequence_index", 0)) / max(n - 1, 1),
            "segment_index": float(seg),
            "duration_s": float(series.get("duration_s", 0.0)),
            "observation_count": float(series.get("observation_count", n)),
        }
        flatten_numeric("esm", obs.get("esm_radar_parameters", {}), feats)
        flatten_numeric("kin", obs.get("approximate_kinematics", {}), feats)
        flatten_numeric("loc", obs.get("estimated_emitter_location", {}), feats)
        feature_rows.append(feats)
        timestamp_iso8601 = obs.get("timestamp_iso8601")
        observation_time_by_node[obs_node_idx] = datetime.fromisoformat(timestamp_iso8601.replace("Z", "+00:00")).astimezone(UTC) if timestamp_iso8601 else None
        node_meta.append({"node_kind": "observation", "observation_id": obs["observation_id"], "series_id": obs["series_id"], "sequence_index": obs["sequence_index"], "timestamp_iso8601": timestamp_iso8601})
        if INCLUDE_INTEL_REPORT_NODES:
            for report_rank, report in enumerate(obs.get("intelligence_reports") or [], start=1):
                pending_report_nodes.append((obs_node_idx, report, report_rank))
        if INCLUDE_CANDIDATE_NODES:
            candidates = obs.get("candidate_labels_from_shared_kg_features") or []
            for rank, candidate in enumerate(candidates, start=1):
                pending_candidate_nodes.append((obs_node_idx, candidate, rank, len(candidates)))

for obs_node_idx, report, report_rank in tqdm(pending_report_nodes, desc="Building report/claim nodes"):
    obs_meta = node_meta[obs_node_idx]
    observation_time = observation_time_by_node.get(obs_node_idx)
    recency = report_recency_score(report, reference_time=observation_time)
    report_idx = len(feature_rows)
    feature_rows.append({
        "node_kind_observation": 0.0,
        "node_kind_candidate": 0.0,
        "node_kind_report": 1.0,
        "node_kind_claim": 0.0,
        "report_rank": float(report_rank),
        "report_credibility_score": float(report.get("credibility_score", 0.5)),
        "report_recency_score": float(recency),
        "report_claim_count": float(len(report.get("claims") or [])),
    })
    node_meta.append({
        "node_kind": "intelligence_report",
        "report_id": report["report_id"],
        "observation_node_idx": obs_node_idx,
        "observation_id": obs_meta["observation_id"],
        "series_id": obs_meta["series_id"],
        "sequence_index": obs_meta["sequence_index"],
    })
    reports_by_observation_node[obs_node_idx].append(report_idx)
    for claim_rank, claim in enumerate(report.get("claims") or [], start=1):
        score = report_claim_score(report, claim, observation_time=observation_time)
        claim_idx = len(feature_rows)
        feature_rows.append({
            "node_kind_observation": 0.0,
            "node_kind_candidate": 0.0,
            "node_kind_report": 0.0,
            "node_kind_claim": 1.0,
            "claim_rank": float(claim_rank),
            "claim_confidence": float(claim.get("claim_confidence", 0.5)),
            "claim_extraction_confidence": float(claim.get("extraction_confidence", 0.5)),
            "claim_specificity_score": float(claim.get("specificity_score", 0.5)),
            "claim_kg_consistency_score": float(claim.get("kg_consistency_score", 0.5)),
            "claim_text_score": float(score),
            "claim_supports": 1.0 if claim.get("stance", "supports") == "supports" else 0.0,
            "claim_refutes": 1.0 if claim.get("stance") == "refutes" else 0.0,
        })
        node_meta.append({
            "node_kind": "report_claim",
            "claim_id": claim["claim_id"],
            "claim_type": claim.get("claim_type"),
            "object_id": claim.get("object_id"),
            "report_node_idx": report_idx,
            "observation_node_idx": obs_node_idx,
            "observation_id": obs_meta["observation_id"],
            "series_id": obs_meta["series_id"],
            "sequence_index": obs_meta["sequence_index"],
        })
        claims_by_observation_node[obs_node_idx].append(claim_idx)
        claims_by_claim_type[(obs_meta["observation_id"], str(claim.get("claim_type")))].append(claim_idx)

for obs_node_idx, candidate, rank, candidate_count in pending_candidate_nodes:
    rank_fraction = float(rank - 1) / max(candidate_count - 1, 1)
    feats = {
        "node_kind_observation": 0.0,
        "node_kind_candidate": 1.0,
        "candidate_rank": float(rank),
        "candidate_rank_fraction": rank_fraction,
        "candidate_count": float(candidate_count),
    }
    candidate_node_idx = len(feature_rows)
    feature_rows.append(feats)
    candidate_id = f"candidate:{node_meta[obs_node_idx]['observation_id']}:{rank}"
    node_meta.append({
        "node_kind": "candidate",
        "candidate_id": candidate_id,
        "observation_node_idx": obs_node_idx,
        "observation_id": node_meta[obs_node_idx]["observation_id"],
        "series_id": node_meta[obs_node_idx]["series_id"],
        "sequence_index": node_meta[obs_node_idx]["sequence_index"],
        "rank": rank,
    })
    candidates_by_observation_node[obs_node_idx].append(candidate_node_idx)

feature_names = sorted({k for row in feature_rows for k in row})
label_like_feature_names = [name for name in feature_names if any(token in name.lower() for token in ("ground_truth", "mode_id", "radar_mode", "aircraft_id", "operator_country", "synthetic_truth"))]
assert not label_like_feature_names, f"Label-like feature names leaked: {label_like_feature_names}"
X_NP_DTYPE = np.float16
TORCH_DATA_DTYPE = torch.float16
X_np = np.asarray([[row.get(k, 0.0) for k in feature_names] for row in feature_rows], dtype=X_NP_DTYPE)
mu, sigma = X_np.astype(np.float32).mean(axis=0), X_np.astype(np.float32).std(axis=0)
sigma[sigma == 0] = 1.0
X = torch.tensor((X_np.astype(np.float32) - mu) / sigma, dtype=TORCH_DATA_DTYPE, device=DEVICE)
print(X.shape, feature_names[:10], {"observation_nodes": len(observation_node_indices), "candidate_nodes": len(pending_candidate_nodes), "report_nodes": sum(1 for m in node_meta if m.get("node_kind") == "intelligence_report"), "claim_nodes": sum(1 for m in node_meta if m.get("node_kind") == "report_claim"), "candidate_nodes_enabled": INCLUDE_CANDIDATE_NODES, "intel_report_nodes_enabled": INCLUDE_INTEL_REPORT_NODES, "segment_edges_enabled": USE_SEGMENT_EDGES})



Building report/claim nodes:   0%|          | 0/240166 [00:00<?, ?it/s]

torch.Size([784949, 72]) ['candidate_count', 'candidate_rank', 'candidate_rank_fraction', 'claim_confidence', 'claim_extraction_confidence', 'claim_kg_consistency_score', 'claim_rank', 'claim_refutes', 'claim_specificity_score', 'claim_supports'] {'observation_nodes': 80118, 'candidate_nodes': 224499, 'report_nodes': 240166, 'claim_nodes': 240166, 'candidate_nodes_enabled': True, 'intel_report_nodes_enabled': True, 'segment_edges_enabled': False}


## Build a relational graph

The default graph contains observation nodes only, with temporal adjacency (`NEXT_OBSERVATION` / `PREV_OBSERVATION`), same-emitter continuity, and self loops. Candidate-evidence and same-segment/mode-shift edges are opt-in ablations because they can encode candidate-hypothesis or mode-derived structure that would inflate a clean radar-mode benchmark.


In [4]:
RELATION_NAMES = ["self", "next_observation", "prev_observation", "same_emitter"]
if USE_SEGMENT_EDGES:
    RELATION_NAMES.extend(["same_mode_segment", "possible_mode_shift"])
if INCLUDE_CANDIDATE_NODES:
    RELATION_NAMES.extend(["has_candidate", "candidate_for", "contradicts_candidate"])
if INCLUDE_INTEL_REPORT_NODES:
    RELATION_NAMES.extend(["has_report", "report_for", "report_contains_claim", "claim_from_report", "claim_supports_observation", "observation_supported_by_claim", "contradicts_claim"])
RELATIONS = {name: idx for idx, name in enumerate(RELATION_NAMES)}
edge_src, edge_dst, edge_type = [], [], []


def add_edge(i, j, rel):
    edge_src.append(i)
    edge_dst.append(j)
    edge_type.append(RELATIONS[rel])


def candidate_contradiction_reasons(left: dict[str, Any], right: dict[str, Any]) -> list[str]:
    comparisons = {"rank": (left.get("rank"), right.get("rank"))}
    return [field for field, (a, b) in comparisons.items() if a is not None and b is not None and a != b]


for i in range(len(node_meta)):
    add_edge(i, i, "self")

series_to_indices = defaultdict(list)
for i in observation_node_indices:
    series_to_indices[node_meta[i]["series_id"]].append(i)

segment_by_node = {i: int(feature_rows[i]["segment_index"]) for i in observation_node_indices}
for indices in series_to_indices.values():
    indices = sorted(indices, key=lambda i: node_meta[i]["sequence_index"])
    for a, b in zip(indices, indices[1:]):
        add_edge(a, b, "next_observation")
        add_edge(b, a, "prev_observation")
        if USE_SEGMENT_EDGES:
            if segment_by_node[a] == segment_by_node[b]:
                add_edge(a, b, "same_mode_segment")
                add_edge(b, a, "same_mode_segment")
            else:
                add_edge(a, b, "possible_mode_shift")
                add_edge(b, a, "possible_mode_shift")
    # Sparse same-emitter reinforcement edges between observations two samples apart.
    for a, b in zip(indices, indices[2:]):
        add_edge(a, b, "same_emitter")
        add_edge(b, a, "same_emitter")

contradiction_reasons = Counter()
if INCLUDE_CANDIDATE_NODES:
    for obs_idx, candidate_indices in candidates_by_observation_node.items():
        candidate_indices = sorted(candidate_indices, key=lambda idx: node_meta[idx]["rank"])
        for candidate_idx in candidate_indices:
            add_edge(obs_idx, candidate_idx, "has_candidate")
            add_edge(candidate_idx, obs_idx, "candidate_for")
        for left_pos, left_idx in enumerate(candidate_indices):
            for right_idx in candidate_indices[left_pos + 1:]:
                reasons = candidate_contradiction_reasons(node_meta[left_idx], node_meta[right_idx])
                if not reasons:
                    continue
                add_edge(left_idx, right_idx, "contradicts_candidate")
                contradiction_reasons.update(reasons)

if INCLUDE_INTEL_REPORT_NODES:
    for obs_idx, report_indices in reports_by_observation_node.items():
        for report_idx in report_indices:
            add_edge(obs_idx, report_idx, "has_report")
            add_edge(report_idx, obs_idx, "report_for")
    for obs_idx, claim_indices in claims_by_observation_node.items():
        for claim_idx in claim_indices:
            report_idx = node_meta[claim_idx]["report_node_idx"]
            add_edge(report_idx, claim_idx, "report_contains_claim")
            add_edge(claim_idx, report_idx, "claim_from_report")
            add_edge(claim_idx, obs_idx, "claim_supports_observation")
            add_edge(obs_idx, claim_idx, "observation_supported_by_claim")
    for claim_indices in claims_by_claim_type.values():
        for left_pos, left_idx in enumerate(claim_indices):
            for right_idx in claim_indices[left_pos + 1:]:
                if node_meta[left_idx].get("object_id") != node_meta[right_idx].get("object_id"):
                    add_edge(left_idx, right_idx, "contradicts_claim")
                    contradiction_reasons.update(["report_claim_object_id"])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_types = torch.tensor(edge_type, dtype=torch.long, device=DEVICE)
print({name: int((edge_types == rid).sum().cpu()) for name, rid in RELATIONS.items()})
print({"contradiction_reasons": dict(contradiction_reasons)})



{'self': 784949, 'next_observation': 77618, 'prev_observation': 77618, 'same_emitter': 150236, 'has_candidate': 224499, 'candidate_for': 224499, 'contradicts_candidate': 286698, 'has_report': 240166, 'report_for': 240166, 'report_contains_claim': 240166, 'claim_from_report': 240166, 'claim_supports_observation': 240166, 'observation_supported_by_claim': 240166, 'contradicts_claim': 0}
{'contradiction_reasons': {'rank': 286698}}


## Encode labels and create the required 0.5 / 0.3 / 0.2 train-test-validation split

Splitting is performed by `series_id`, not by individual observation, so adjacent observations from the same emitter track cannot cross from training into evaluation splits. Series are stratified by the joint `(aircraft_variant, operator_country)` label so both aircraft variants and operators remain represented proportionally across the 0.5 / 0.3 / 0.2 train-test-validation partitions.


In [5]:
label_vocab, y = {}, {}
for task in TARGETS:
    vals = [row[task] for row in target_rows]
    classes = sorted(set(vals))
    label_vocab[task] = classes
    idx = {v: i for i, v in enumerate(classes)}
    y[task] = torch.tensor([idx[v] for v in vals], dtype=torch.long, device=DEVICE)
    print(task, len(classes), Counter(vals).most_common(3))

def series_stratification_labels() -> dict[str, tuple[str, str]]:
    """Return one (aircraft_variant, operator_country) stratum per series."""
    labels_by_series: dict[str, set[tuple[str, str]]] = defaultdict(set)
    for row in target_rows:
        labels_by_series[row["series_id"]].add((row["aircraft_variant"], row["operator_country"]))

    inconsistent = {series_id: labels for series_id, labels in labels_by_series.items() if len(labels) != 1}
    if inconsistent:
        examples = list(inconsistent.items())[:3]
        raise ValueError(f"Expected a single aircraft/operator stratum per series; examples: {examples}")
    return {series_id: next(iter(labels)) for series_id, labels in labels_by_series.items()}


def stratified_series_split(
    labels_by_series: dict[str, tuple[str, str]],
    fractions: dict[str, float],
    seed: int,
) -> dict[str, set[str]]:
    """Split series IDs while preserving joint aircraft-variant/operator strata."""
    split_names = tuple(fractions)
    desired = {name: fractions[name] * len(labels_by_series) for name in split_names}
    split_series = {name: set() for name in split_names}
    strata: dict[tuple[str, str], list[str]] = defaultdict(list)
    for series_id, stratum in labels_by_series.items():
        strata[stratum].append(series_id)

    rng = np.random.default_rng(seed)
    stratum_items = list(strata.items())
    rng.shuffle(stratum_items)
    # Assign larger strata first so common aircraft/operator pairs closely follow the requested ratios.
    stratum_items.sort(key=lambda item: len(item[1]), reverse=True)

    for _, ids in stratum_items:
        ids = list(ids)
        rng.shuffle(ids)
        quotas = {name: fractions[name] * len(ids) for name in split_names}
        assigned = {name: 0 for name in split_names}
        for series_id in ids:
            split_name = max(
                split_names,
                key=lambda name: (quotas[name] - assigned[name], desired[name] - len(split_series[name])),
            )
            split_series[split_name].add(series_id)
            assigned[split_name] += 1

    return split_series


series_labels = series_stratification_labels()
split_series = stratified_series_split(series_labels, {"train": 0.5, "test": 0.3, "val": 0.2}, SEED)
splits = {
    name: torch.tensor([i for i in observation_node_indices if node_meta[i]["series_id"] in ids], dtype=torch.long, device=DEVICE)
    for name, ids in split_series.items()
}
print({k: len(v) for k, v in splits.items()})
print({k: len(v) for k, v in split_series.items()})
for split_name, ids in split_series.items():
    split_strata = Counter(series_labels[series_id] for series_id in ids)
    print(split_name, "aircraft/operator strata", len(split_strata), split_strata.most_common(3))


aircraft_variant 66 [('F-15SA', 1565), ('J-10A', 1543), ('J-8F', 1538)]
radar_mode 5 [('air_to_ground_mapping', 16537), ('look_down_shoot_down', 16136), ('track_while_scan', 16106)]
radar_type 48 [('RBE2-AA', 5071), ('AN/APG-79', 4136), ('AN/APG-81', 3763)]
operator_country 58 [('India', 13972), ('United States', 9784), ('China', 9776)]
{'train': 39775, 'test': 24326, 'val': 16017}
{'train': 1257, 'test': 745, 'val': 498}
train aircraft/operator strata 184 [(('Su-30MKI', 'India'), 24), (('J-10A', 'China'), 24), (('J-8F', 'China'), 24)]
test aircraft/operator strata 182 [(('Su-30MKI', 'India'), 14), (('J-8F', 'China'), 14), (('Mirage 2000I', 'India'), 14)]
val aircraft/operator strata 175 [(('J-8F', 'China'), 10), (('J-16', 'China'), 9), (('Mirage 2000I', 'India'), 9)]


## Define a configurable-hop GraphSAGE encoder with relation-aware HGT attention

The classifier below replaces the residual r-GCN stack with two stages. First, a GraphSAGE-style neighbourhood sampler/aggregator repeatedly pools inbound neighbours so information can travel over a user-configurable number of graph edges. `NUM_MESSAGE_PASSING_EDGES` defaults to `7`, satisfying the requirement that the model can pass messages over at least seven edges; increase it to propagate across longer paths.

Second, relation-aware HGT layers apply multi-head attention over typed edges. Each relation has learned key/value/attention priors, so report-support, candidate, temporal, and self-loop edges can contribute differently while still sharing the same hidden node representation.


In [6]:
HIDDEN_DIM = 32
NUM_MESSAGE_PASSING_EDGES = 7# user-configurable GraphSAGE hops; keep >= 7 for seven-edge receptive fields
NUM_HGT_LAYERS = 1
NUM_ATTENTION_HEADS = 4
DROPOUT = 0.25
TASK_HEAD_HIDDEN_DIM = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
L1_LAMBDA = 1e-5
PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
BATCH_SIZE = 50_000
EDGE_CHUNK_SIZE = 20_000_000  # caps temporary per-edge message tensors to reduce GPU peak memory

# Keep trainable parameters in FP32. GradScaler cannot unscale FP16 gradients;
# autocast below still runs eligible CUDA ops in FP16 for memory savings.
MODEL_DTYPE = torch.float32
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
GRADIENT_CHECKPOINTING = True

if NUM_MESSAGE_PASSING_EDGES < 1:
    raise ValueError("NUM_MESSAGE_PASSING_EDGES must be at least 5 so messages can traverse seven graph edges.")

class NeighborSamplingGraphSAGEBlock(nn.Module):
    """GraphSAGE mean aggregation over inbound neighbours with optional deterministic fanout."""
    def __init__(self, hidden_dim: int, dropout: float, fanout: int | None = None, edge_chunk_size: int = 0):
        super().__init__()
        self.self_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.neighbor_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fanout = None if fanout is None or fanout <= 0 else int(fanout)
        if edge_chunk_size < 0:
            raise ValueError("edge_chunk_size must be non-negative")
        self.edge_chunk_size = int(edge_chunk_size)

    def _sample_edges(self, src, dst):
        if self.fanout is None or src.numel() == 0:
            return src, dst
        keep = torch.zeros(src.numel(), device=src.device, dtype=torch.bool)
        for node in torch.unique(dst):
            positions = torch.nonzero(dst == node, as_tuple=False).flatten()
            keep[positions[: self.fanout]] = True
        return src[keep], dst[keep]

    def forward(self, x, edge_index, edge_types=None):
        if edge_index.numel() == 0:
            return self.norm(x + self.dropout(F.gelu(self.self_projection(x))))
        src, dst = self._sample_edges(edge_index[0], edge_index[1])
        aggregated = torch.zeros_like(x)
        degree = torch.zeros(x.size(0), device=x.device, dtype=x.dtype)
        if self.edge_chunk_size > 0:
            for start in range(0, int(src.numel()), self.edge_chunk_size):
                stop = start + self.edge_chunk_size
                aggregated.index_add_(0, dst[start:stop], x[src[start:stop]])
                degree.index_add_(0, dst[start:stop], torch.ones(min(self.edge_chunk_size, src.numel() - start), device=x.device, dtype=x.dtype))
        else:
            aggregated.index_add_(0, dst, x[src])
            degree.index_add_(0, dst, torch.ones(dst.numel(), device=x.device, dtype=x.dtype))
        aggregated = aggregated / degree.clamp_min(1).unsqueeze(-1)
        updated = self.self_projection(x) + self.neighbor_projection(aggregated)
        updated = self.dropout(F.gelu(updated))
        return self.norm(x + updated)

class RelationAwareHGTLayer(nn.Module):
    """HGT-style relation-aware multi-head attention over typed graph edges."""
    def __init__(self, hidden_dim: int, num_relations: int, num_heads: int, dropout: float, edge_chunk_size: int = 0):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")
        self.num_relations = max(num_relations, 1)
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.relation_key = nn.Parameter(torch.empty(self.num_relations, num_heads, self.head_dim, self.head_dim))
        self.relation_value = nn.Parameter(torch.empty(self.num_relations, num_heads, self.head_dim, self.head_dim))
        self.relation_priority = nn.Parameter(torch.ones(self.num_relations, num_heads))
        self.output = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        if edge_chunk_size < 0:
            raise ValueError("edge_chunk_size must be non-negative")
        self.edge_chunk_size = int(edge_chunk_size)
        nn.init.xavier_uniform_(self.relation_key)
        nn.init.xavier_uniform_(self.relation_value)

    def _edge_messages(self, x, src, dst, rel):
        q = self.query(x[dst]).view(-1, self.num_heads, self.head_dim)
        k = self.key(x[src]).view(-1, self.num_heads, self.head_dim)
        v = self.value(x[src]).view(-1, self.num_heads, self.head_dim)
        rel_k = self.relation_key[rel]
        rel_v = self.relation_value[rel]
        k = torch.einsum("ehd,ehdf->ehf", k, rel_k)
        v = torch.einsum("ehd,ehdf->ehf", v, rel_v)
        scores = (q * k).sum(dim=-1) / math.sqrt(self.head_dim)
        scores = scores * self.relation_priority[rel]
        # Per-destination sigmoid gates keep memory bounded without requiring dense per-node softmax buffers.
        weights = torch.sigmoid(scores).unsqueeze(-1)
        return (weights * v).reshape(src.numel(), -1)

    def forward(self, x, edge_index, edge_types):
        if edge_index.numel() == 0:
            return x
        src, dst = edge_index
        messages = torch.zeros_like(x)
        degree = torch.zeros(x.size(0), device=x.device, dtype=x.dtype)
        for rel_id in range(self.num_relations):
            mask = edge_types == rel_id
            if not torch.any(mask):
                continue
            rel_src, rel_dst = src[mask], dst[mask]
            if self.edge_chunk_size > 0:
                for start in range(0, int(rel_src.numel()), self.edge_chunk_size):
                    stop = start + self.edge_chunk_size
                    chunk_src, chunk_dst = rel_src[start:stop], rel_dst[start:stop]
                    chunk_rel = torch.full_like(chunk_src, rel_id)
                    messages.index_add_(0, chunk_dst, self._edge_messages(x, chunk_src, chunk_dst, chunk_rel).to(messages.dtype))
                    degree.index_add_(0, chunk_dst, torch.ones(chunk_dst.numel(), device=x.device, dtype=x.dtype))
            else:
                rel_tensor = torch.full_like(rel_src, rel_id)
                messages.index_add_(0, rel_dst, self._edge_messages(x, rel_src, rel_dst, rel_tensor).to(messages.dtype))
                degree.index_add_(0, rel_dst, torch.ones(rel_dst.numel(), device=x.device, dtype=x.dtype))
        messages = messages / degree.clamp_min(1).unsqueeze(-1)
        updated = self.output(messages)
        updated = self.dropout(F.gelu(updated))
        return self.norm(x + updated)

class SeriesGraphSAGEHGTClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_relations, class_sizes, num_message_passing_edges=7, num_hgt_layers=2, num_heads=4, dropout=0.2, task_head_hidden_dim=128, sage_fanout=None, edge_chunk_size=0, gradient_checkpointing=False):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.sage_layers = nn.ModuleList([NeighborSamplingGraphSAGEBlock(hidden_dim, dropout, fanout=sage_fanout, edge_chunk_size=edge_chunk_size) for _ in range(num_message_passing_edges)])
        self.hgt_layers = nn.ModuleList([RelationAwareHGTLayer(hidden_dim, num_relations, num_heads, dropout, edge_chunk_size=edge_chunk_size) for _ in range(num_hgt_layers)])
        self.gradient_checkpointing = bool(gradient_checkpointing)
        self.heads = nn.ModuleDict({task: nn.Sequential(nn.Linear(hidden_dim, task_head_hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(task_head_hidden_dim, size)) for task, size in class_sizes.items()})

    def encode(self, x, edge_index, edge_types):
        h = self.input_projection(x)
        for layer in self.sage_layers:
            h = torch.utils.checkpoint.checkpoint(layer, h, edge_index, edge_types, use_reentrant=False) if self.gradient_checkpointing and self.training else layer(h, edge_index, edge_types)
        for layer in self.hgt_layers:
            h = torch.utils.checkpoint.checkpoint(layer, h, edge_index, edge_types, use_reentrant=False) if self.gradient_checkpointing and self.training else layer(h, edge_index, edge_types)
        return h

    def forward(self, x, edge_index, edge_types):
        h = self.encode(x, edge_index, edge_types)
        return {task: head(h) for task, head in self.heads.items()}

model_config = {
    "hidden_dim": HIDDEN_DIM,
    "num_message_passing_edges": NUM_MESSAGE_PASSING_EDGES,
    "num_hgt_layers": NUM_HGT_LAYERS,
    "num_attention_heads": NUM_ATTENTION_HEADS,
    "dropout": DROPOUT,
    "task_head_hidden_dim": TASK_HEAD_HIDDEN_DIM,
    "l1_lambda": L1_LAMBDA,
    "patience": PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "batch_size": BATCH_SIZE,
    "edge_chunk_size": EDGE_CHUNK_SIZE,
    "model_dtype": str(MODEL_DTYPE),
    "data_dtype": str(TORCH_DATA_DTYPE),
    "numpy_ingest_dtype": str(X_NP_DTYPE),
    "use_amp": USE_AMP,
    "amp_dtype": str(AMP_DTYPE),
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "include_candidate_nodes": INCLUDE_CANDIDATE_NODES,
    "use_segment_edges": USE_SEGMENT_EDGES,
}
model = SeriesGraphSAGEHGTClassifier(
    X.size(1), HIDDEN_DIM, len(RELATIONS), {t: len(label_vocab[t]) for t in TARGETS},
    num_message_passing_edges=NUM_MESSAGE_PASSING_EDGES,
    num_hgt_layers=NUM_HGT_LAYERS,
    num_heads=NUM_ATTENTION_HEADS,
    dropout=DROPOUT,
    task_head_hidden_dim=TASK_HEAD_HIDDEN_DIM,
    edge_chunk_size=EDGE_CHUNK_SIZE,
    sage_fanout=8,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
).to(device=DEVICE, dtype=MODEL_DTYPE)
if USE_AMP and DEVICE.type == "cuda" and next(model.parameters()).dtype == torch.float16:
    raise ValueError("Keep MODEL_DTYPE at torch.float32 when using GradScaler; autocast controls FP16 compute.")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)
print(model)
print(model_config)


SeriesGraphSAGEHGTClassifier(
  (input_projection): Sequential(
    (0): Linear(in_features=72, out_features=32, bias=True)
    (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.25, inplace=False)
  )
  (sage_layers): ModuleList(
    (0-6): 7 x NeighborSamplingGraphSAGEBlock(
      (self_projection): Linear(in_features=32, out_features=32, bias=False)
      (neighbor_projection): Linear(in_features=32, out_features=32, bias=False)
      (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.25, inplace=False)
    )
  )
  (hgt_layers): ModuleList(
    (0): RelationAwareHGTLayer(
      (query): Linear(in_features=32, out_features=32, bias=True)
      (key): Linear(in_features=32, out_features=32, bias=True)
      (value): Linear(in_features=32, out_features=32, bias=True)
      (output): Linear(in_features=32, out_features=32, bias=True)
      (norm): LayerNorm((32,), eps=1e-05, elementwise_

In [ ]:
# Estimate edge-computation demand and EDGE_CHUNK_SIZE ceiling
#
# This model performs full-graph message passing for every mini-batch loss evaluation,
# so BATCH_SIZE changes how many supervised nodes contribute to each optimizer step,
# but it does not reduce the number of graph-edge computations in that step.

from collections import defaultdict


def estimate_edge_computation_requirements(edge_index, edge_types, relation_names, *, num_sage_layers, num_hgt_layers, sage_fanout=None, batch_size=None):
    edge_count = int(edge_index.size(1))
    node_count = int(edge_index.max().item() + 1) if edge_index.numel() else 0

    sampled_sage_edge_count = edge_count
    if sage_fanout is not None and sage_fanout > 0 and edge_count:
        # Mirror NeighborSamplingGraphSAGEBlock._sample_edges: keep up to the first
        # sage_fanout inbound edges for each destination node.
        inbound_seen_by_dst = defaultdict(int)
        kept_edges = 0
        for dst in edge_index[1].detach().cpu().tolist():
            if inbound_seen_by_dst[dst] < sage_fanout:
                kept_edges += 1
            inbound_seen_by_dst[dst] += 1
        sampled_sage_edge_count = kept_edges

    relation_edge_counts = {
        relation_name: int((edge_types == relation_id).sum().detach().cpu())
        for relation_name, relation_id in relation_names.items()
    }
    max_hgt_relation_edges = max(relation_edge_counts.values(), default=0)
    maximum_edges_processed_without_chunking = max(sampled_sage_edge_count, max_hgt_relation_edges)
    total_edge_computations_per_forward = (num_sage_layers * sampled_sage_edge_count) + (num_hgt_layers * edge_count)

    return {
        "node_count": node_count,
        "edge_count": edge_count,
        "batch_size": None if batch_size is None else int(batch_size),
        "num_graphsage_layers": int(num_sage_layers),
        "num_hgt_layers": int(num_hgt_layers),
        "sage_fanout": None if sage_fanout is None else int(sage_fanout),
        "sampled_graphsage_edges_per_layer": sampled_sage_edge_count,
        "max_hgt_relation_edges_per_layer": max_hgt_relation_edges,
        "maximum_required_edge_chunk_size": maximum_edges_processed_without_chunking,
        "total_edge_computations_per_forward": total_edge_computations_per_forward,
        "total_edge_computations_per_training_batch": total_edge_computations_per_forward,
        "note": "Full-graph forward pass: BATCH_SIZE affects supervised loss nodes, not message-passing edge count.",
        "relation_edge_counts": relation_edge_counts,
    }


edge_compute_requirements = estimate_edge_computation_requirements(
    edge_index,
    edge_types,
    RELATIONS,
    num_sage_layers=NUM_MESSAGE_PASSING_EDGES,
    num_hgt_layers=NUM_HGT_LAYERS,
    sage_fanout=8,
    batch_size=BATCH_SIZE,
)

for key, value in edge_compute_requirements.items():
    if key == "relation_edge_counts":
        continue
    print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")
print("relation_edge_counts:")
for relation_name, count in edge_compute_requirements["relation_edge_counts"].items():
    print(f"  {relation_name}: {count:,}")
print(
    "Set EDGE_CHUNK_SIZE >= "
    f"{edge_compute_requirements['maximum_required_edge_chunk_size']:,} "
    "to process every current layer/relation without chunking."
)



## Train with mini-batches, checkpoints, per-epoch diagnostics, and TensorBoard logs

Training uses a `DataLoader` over the training observation-node indices so the supervised loss is optimized in configurable mini-batches. Each mini-batch still runs message passing over the full relational graph to preserve temporal, same-emitter, segment, and candidate-evidence context; only the supervised loss indices are batched. The L1 penalty is scaled by the mini-batch fraction so the total regularization strength remains comparable to full-batch training across one epoch.


In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
except Exception:
    SummaryWriter = None

def classification_loss(logits, indices):
    return sum(F.cross_entropy(logits[task][indices], y[task][indices]) for task in TARGETS)

def l1_penalty(model: nn.Module) -> torch.Tensor:
    """Return the L1 norm across all trainable model parameters."""
    return sum(parameter.abs().sum() for parameter in model.parameters())

def accuracy(logits, labels):
    return float((logits.argmax(dim=-1) == labels).float().mean().detach().cpu())

@torch.no_grad()
def split_metrics(split):
    model.eval(); idx = splits[split]
    with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
        logits = model(X, edge_index, edge_types)
    metrics = {"loss": float(classification_loss(logits, idx).detach().cpu())}
    for task in TARGETS:
        metrics[f"{task}_acc"] = accuracy(logits[task][idx], y[task][idx])
    return metrics

def make_train_loader():
    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_indices = splits["train"].detach().cpu()
    return DataLoader(train_indices, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

EPOCHS = 200
ckpt_path = ARTIFACT_DIR / "best_model.pt"
writer = SummaryWriter(str(ARTIFACT_DIR / "tensorboard")) if SummaryWriter else None
history, best_val, bad_epochs = [], math.inf, 0
train_loader = make_train_loader()
train_size = max(int(splits["train"].numel()), 1)
batches_per_epoch = len(train_loader)
print(f"training observations={train_size:,}, batch_size={BATCH_SIZE:,}, batches_per_epoch={batches_per_epoch:,}")

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        weighted_train_data_loss = 0.0
        weighted_l1_penalty = 0.0
        batch_progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch}/{EPOCHS}",
            total=batches_per_epoch,
            unit="batch",
            leave=True,
            dynamic_ncols=True,
        )
        for batch_indices in batch_progress:
            batch_indices = batch_indices.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
                logits = model(X, edge_index, edge_types)
                data_loss = classification_loss(logits, batch_indices)
                batch_fraction = batch_indices.numel() / train_size
                regularization = L1_LAMBDA * l1_penalty(model) * batch_fraction
                loss = data_loss + regularization
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            weight = batch_indices.numel() / train_size
            weighted_train_data_loss += float(data_loss.detach().cpu()) * weight
            weighted_l1_penalty += float(regularization.detach().cpu())
            batch_progress.set_postfix(
                data_loss=f"{float(data_loss.detach().cpu()):.4f}",
                l1=f"{float(regularization.detach().cpu()):.4f}",
            )
        row = {"epoch": epoch, "train_data_loss": weighted_train_data_loss, "l1_penalty": weighted_l1_penalty}
        for split in ("train", "test", "val"):
            m = split_metrics(split)
            row.update({f"{split}_{k}": v for k, v in m.items()})
            if writer:
                writer.add_scalar(f"loss/{split}", m["loss"], epoch)
                for task in TARGETS: writer.add_scalar(f"accuracy/{split}_{task}", m[f"{task}_acc"], epoch)
        if writer:
            writer.add_scalar("regularization/l1_penalty", row["l1_penalty"], epoch)
            writer.add_scalar("loss/train_total_with_l1", row["train_data_loss"] + row["l1_penalty"], epoch)
        history.append(row)
        if row["val_loss"] < best_val - EARLY_STOPPING_MIN_DELTA:
            best_val = row["val_loss"]
            bad_epochs = 0
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "label_vocab": label_vocab,
                "feature_names": feature_names,
                "model_config": model_config,
                "splits": {k: v.detach().cpu().tolist() for k, v in splits.items()},
                "history": history,
                "best_val_loss": best_val,
            }, ckpt_path)
        else:
            bad_epochs += 1
        diag = [f"epoch={epoch:04d}", f"train_loss={row['train_loss']:.4f}", f"test_loss={row['test_loss']:.4f}", f"val_loss={row['val_loss']:.4f}", f"batch_train_loss={row['train_data_loss']:.4f}", f"l1={row['l1_penalty']:.4f}", f"bad_epochs={bad_epochs}/{PATIENCE}"]
        diag += [f"train_{task}_acc={row[f'train_{task}_acc']:.3f}" for task in TARGETS]
        diag += [f"test_{task}_acc={row[f'test_{task}_acc']:.3f}" for task in TARGETS]
        print(" | ".join(diag))
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
finally:
    if writer: writer.close()
print(f"best checkpoint: {ckpt_path} (val_loss={best_val:.4f})")


training observations=39,775, batch_size=50,000, batches_per_epoch=1


Epoch 1/200:   0%|          | 0/1 [00:00<?, ?batch/s]

## Final evaluation, reports, predictions, and plots


In [ ]:
checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
final_metrics = {split: split_metrics(split) for split in ("train", "test", "val")}

model.eval()
with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
    final_logits = model(X, edge_index, edge_types)
    probs = {task: F.softmax(final_logits[task], dim=-1).detach().cpu().numpy() for task in TARGETS}

predictions = []
for target_idx, node_idx in enumerate(observation_node_indices):
    meta = node_meta[node_idx]
    rec = dict(meta)
    for task in TARGETS:
        pred_idx = int(probs[task][node_idx].argmax())
        rec[f"true_{task}"] = target_rows[target_idx][task]
        rec[f"pred_{task}"] = label_vocab[task][pred_idx]
        rec[f"pred_{task}_confidence"] = float(probs[task][node_idx][pred_idx])
    predictions.append(rec)

summary = {"final_metrics": final_metrics, "best_epoch": int(checkpoint["epoch"]), "split_sizes": {k: int(v.numel()) for k, v in splits.items()}, "split_series_counts": {k: len(v) for k, v in split_series.items()}, "targets": TARGETS, "relations": RELATIONS, "node_counts": {"observations": len(observation_node_indices), "candidates": sum(1 for m in node_meta if m.get("node_kind") == "candidate"), "intelligence_reports": sum(1 for m in node_meta if m.get("node_kind") == "intelligence_report"), "report_claims": sum(1 for m in node_meta if m.get("node_kind") == "report_claim")}, "model_config": checkpoint.get("model_config", model_config), "best_val_loss": float(checkpoint.get("best_val_loss", final_metrics["val"]["loss"])), "leakage_guard": "ground truth keys stripped; supervised nodes are observations only; candidate nodes/edges and segment edges disabled by default; intelligence report truth markers stripped"}
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))



In [ ]:
epochs = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for split in ("train", "test", "val"):
    axes[0].plot(epochs, [r[f"{split}_loss"] for r in history], label=split)
axes[0].set_title("Classification loss by split")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("sum cross-entropy"); axes[0].legend(); axes[0].grid(True, alpha=.3)
for task in TARGETS:
    axes[1].plot(epochs, [r[f"test_{task}_acc"] for r in history], label=f"test {task}")
axes[1].set_title("Test accuracy by task")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=.3)
fig.tight_layout()
plot_path = ARTIFACT_DIR / "training_metrics.png"
fig.savefig(plot_path, dpi=150)
plot_path
